# FEASE Model Evaluation

This notebook shows how to measure recommendation quality for a trained FEASE model. We build a synthetic dataset with latent persona structure, hold out a fraction of each user's interactions, train on the rest, and report standard ranking metrics plus catalog-level signals.

**Metrics covered**

- Recall@K, Precision@K, Hit Rate@K
- NDCG@K, MAP@K
- Warm vs cold-start user breakdown
- Catalog coverage, intra-list genre diversity, popularity-based novelty
- Lambda sweep (L2 regularization) with NDCG@10 vs regularization strength

**Prereq.** Run `.venv/bin/maturin develop` first so `cr_fease` is importable.

In [ ]:
import math
import tempfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import polars as pl

import cr_fease as fease

RNG = np.random.default_rng(seed=2026)
WORK_DIR = Path(tempfile.mkdtemp(prefix="fease_eval_"))
print(f"Working dir: {WORK_DIR}")

## 1. Synthetic dataset with latent structure

We simulate 500 users across 5 personas, each preferring a primary and secondary genre. Items are assigned to one of 5 genres. This gives a ground-truth relevance signal the model can plausibly recover.

We also add a small fraction of cold-start users (present only in `user_features`) so we can measure cold-start quality.

In [ ]:
N_USERS = 500
N_COLD_USERS = 50      # extra users that will appear only in features
N_ITEMS = 120
GENRES = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
PLANS = ["Free", "Premium"]
DEVICES = ["Mobile", "Web", "Console", "TV"]
REGIONS = ["US", "EMEA", "APAC"]

PERSONAS = {
    "action":  ("Action", "SciFi"),
    "comedy":  ("Comedy", "Romance"),
    "drama":   ("Drama", "Romance"),
    "scifi":   ("SciFi", "Action"),
    "romance": ("Romance", "Comedy"),
}

# --- items ---
item_ids = [f"I{i:04d}" for i in range(N_ITEMS)]
item_genre = {iid: GENRES[i % len(GENRES)] for i, iid in enumerate(item_ids)}
item_type = {iid: ("movie" if i % 3 == 0 else "episode") for i, iid in enumerate(item_ids)}
item_feat_rows = []
for iid in item_ids:
    item_feat_rows.append((iid, f"genre_{item_genre[iid]}", 1.0))
    item_feat_rows.append((iid, f"type_{item_type[iid]}", 1.0))
df_items = pl.DataFrame(item_feat_rows, schema=["item_id", "feature_name", "value"], orient="row")

# --- warm users (with interactions) ---
persona_names = list(PERSONAS.keys())
warm_user_ids = [f"U{i:05d}" for i in range(N_USERS)]
cold_user_ids = [f"C{i:05d}" for i in range(N_COLD_USERS)]
all_user_ids = warm_user_ids + cold_user_ids

user_persona = {uid: persona_names[RNG.integers(0, len(persona_names))] for uid in all_user_ids}
user_plan = {uid: PLANS[RNG.integers(0, len(PLANS))] for uid in all_user_ids}
user_device = {uid: DEVICES[RNG.integers(0, len(DEVICES))] for uid in all_user_ids}
user_region = {uid: REGIONS[RNG.integers(0, len(REGIONS))] for uid in all_user_ids}

user_feat_rows = []
for uid in all_user_ids:
    user_feat_rows.append((uid, f"plan_{user_plan[uid]}", 1.0))
    user_feat_rows.append((uid, f"device_{user_device[uid]}", 1.0))
    user_feat_rows.append((uid, f"region_{user_region[uid]}", 1.0))
df_user_features = pl.DataFrame(user_feat_rows, schema=["user_id", "feature_name", "value"], orient="row")

# --- interactions (warm users only) ---
inter_rows = []
for uid in warm_user_ids:
    primary, secondary = PERSONAS[user_persona[uid]]
    weights = np.array([
        4.0 if item_genre[iid] == primary
        else 2.0 if item_genre[iid] == secondary
        else 0.3
        for iid in item_ids
    ])
    probs = weights / weights.sum()
    n_watched = int(RNG.integers(10, 30))
    picks = RNG.choice(item_ids, size=n_watched, replace=False, p=probs)
    for iid in picks:
        inter_rows.append((uid, iid, float(RNG.uniform(1.0, 5.0))))
df_interactions = pl.DataFrame(inter_rows, schema=["user_id", "item_id", "value"], orient="row")

print(f"{df_interactions.height} interactions, {N_USERS} warm + {N_COLD_USERS} cold users, {N_ITEMS} items")

## 2. Train/test split

For each warm user with at least 4 interactions, we hold out 25% (at least 1) as a test set and train on the rest. Cold users contribute 0 training interactions by construction — good for measuring cold-start quality.

In [ ]:
TEST_FRACTION = 0.25
MIN_HISTORY_FOR_SPLIT = 4

train_rows, test_by_user = [], {}
for uid, group in df_interactions.group_by("user_id"):
    uid = uid[0]
    rows = group.select(["item_id", "value"]).to_numpy()
    n = len(rows)
    if n < MIN_HISTORY_FOR_SPLIT:
        for item_id, val in rows:
            train_rows.append((uid, item_id, float(val)))
        continue
    n_test = max(1, int(round(n * TEST_FRACTION)))
    shuffled_idx = RNG.permutation(n)
    test_idx = set(shuffled_idx[:n_test].tolist())
    test_items = []
    for i, (item_id, val) in enumerate(rows):
        if i in test_idx:
            test_items.append(item_id)
        else:
            train_rows.append((uid, item_id, float(val)))
    test_by_user[uid] = set(test_items)

df_train = pl.DataFrame(train_rows, schema=["user_id", "item_id", "value"], orient="row")

i_train_path = WORK_DIR / "train_interactions.parquet"
u_path = WORK_DIR / "user_features.parquet"
t_path = WORK_DIR / "item_features.parquet"
df_train.write_parquet(i_train_path)
df_user_features.write_parquet(u_path)
df_items.write_parquet(t_path)

print(f"Train interactions: {df_train.height}")
print(f"Test users: {len(test_by_user)}  (avg {np.mean([len(v) for v in test_by_user.values()]):.1f} held-out items each)")

## 3. Train

In [ ]:
model = fease.build_and_train(
    interactions_path=str(i_train_path),
    user_features_path=str(u_path),
    item_features_path=str(t_path),
    alpha=1.0, beta=1.0, lambda_=50.0, meta_weight=0.0,
)
print(f"Model: items={model.num_items} user_feats={model.num_user_features} item_feats={model.num_item_features}")

## 4. Ranking metrics

All metrics follow standard top-K definitions:

- **Recall@K** — fraction of the user's held-out items that appear in the top-K.
- **Precision@K** — fraction of the top-K that are held-out items.
- **Hit Rate@K** — 1 if at least one held-out item appears in the top-K, else 0.
- **NDCG@K** — Normalized Discounted Cumulative Gain; rewards placing relevant items near the top.
- **MAP@K** — mean of per-query average precision truncated at K.

In [ ]:
def recall_at_k(ranked, relevant, k):
    if not relevant:
        return None
    return len(set(ranked[:k]) & relevant) / len(relevant)

def precision_at_k(ranked, relevant, k):
    if not relevant:
        return None
    return len(set(ranked[:k]) & relevant) / k

def hit_rate_at_k(ranked, relevant, k):
    if not relevant:
        return None
    return 1.0 if (set(ranked[:k]) & relevant) else 0.0

def ndcg_at_k(ranked, relevant, k):
    if not relevant:
        return None
    dcg = 0.0
    for i, item in enumerate(ranked[:k]):
        if item in relevant:
            dcg += 1.0 / math.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0

def average_precision_at_k(ranked, relevant, k):
    if not relevant:
        return None
    hits, ap = 0, 0.0
    for i, item in enumerate(ranked[:k]):
        if item in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k)

## 5. Evaluate the trained model

For each user with held-out items, we score the full catalog and compute metrics at K=5, 10, 20. Items the user already saw in training are filtered out by `predict` automatically.

In [ ]:
# Build lookup dicts so we can feed the model each user's training history + features
train_hist_by_user = defaultdict(dict)
for uid, iid, val in df_train.iter_rows():
    train_hist_by_user[uid][iid] = float(val)

feats_by_user = defaultdict(dict)
for uid, fname, val in df_user_features.iter_rows():
    feats_by_user[uid][fname] = float(val)

KS = (5, 10, 20)

def evaluate(model, users):
    rows = []
    for uid in users:
        relevant = test_by_user.get(uid, set())
        if not relevant:
            continue
        inter = train_hist_by_user.get(uid, {})
        feats = feats_by_user.get(uid, {})
        recs = model.predict(inter, feats, top_k=max(KS))
        ranked = [guid for guid, _ in recs]
        row = {"user_id": uid, "history_size": len(inter)}
        for k in KS:
            row[f"recall@{k}"] = recall_at_k(ranked, relevant, k)
            row[f"precision@{k}"] = precision_at_k(ranked, relevant, k)
            row[f"hit@{k}"] = hit_rate_at_k(ranked, relevant, k)
            row[f"ndcg@{k}"] = ndcg_at_k(ranked, relevant, k)
            row[f"map@{k}"] = average_precision_at_k(ranked, relevant, k)
        rows.append(row)
    return pl.DataFrame(rows)

eval_df = evaluate(model, list(test_by_user.keys()))

metric_cols = [c for c in eval_df.columns if c not in ("user_id", "history_size")]
summary = eval_df.select([pl.col(c).mean().alias(c) for c in metric_cols])
print(f"Evaluated {eval_df.height} users\n")
for c in metric_cols:
    print(f"  {c:<14s} = {summary[c][0]:.4f}")

## 6. Warm vs cold-start breakdown

Cold-start performance is the payoff of FEASE over vanilla EASE. We measure two groups:

- **warm** — users with ≥ 5 training interactions.
- **light** — 1–4 training interactions (partially cold).

We also train on a version of the dataset that includes the pure cold users (`C*` ids, in features only) and measure how well the model serves them using only features.

In [ ]:
warm_users = [u for u in test_by_user if len(train_hist_by_user.get(u, {})) >= 5]
light_users = [u for u in test_by_user if 0 < len(train_hist_by_user.get(u, {})) < 5]
print(f"warm users: {len(warm_users)}, light users: {len(light_users)}")

for label, users in [("warm", warm_users), ("light", light_users)]:
    if not users:
        continue
    df = evaluate(model, users)
    print(f"\n[{label}]  n={df.height}")
    for c in ("recall@10", "ndcg@10", "hit@10", "map@10"):
        print(f"  {c:<10s} = {df[c].mean():.4f}")

In [ ]:
# Pure cold-start evaluation: score the C* users using features only.
# We don't have held-out interactions for them, but we can inspect whether the
# top recommendations align with the persona's preferred genres (a qualitative signal).
cold_sample = cold_user_ids[:5]
for uid in cold_sample:
    primary, secondary = PERSONAS[user_persona[uid]]
    recs = model.predict({}, feats_by_user[uid], top_k=10)
    top_genres = Counter(item_genre[g] for g, _ in recs)
    print(f"{uid} persona={user_persona[uid]:<8s} (pref={primary}/{secondary})  top10 genres={dict(top_genres)}")

## 7. Catalog-level metrics

Ranking metrics don't tell you if the model is recommending only a popular sliver of the catalog. These help:

- **Coverage@K** — fraction of items that appear in at least one user's top-K.
- **Intra-list diversity** — average number of distinct genres among a user's top-K (higher = more varied within a single list).
- **Novelty** — mean self-information `-log2(popularity)` of recommended items. Higher = less head-heavy.

In [ ]:
TOP_K = 10
recommended_items = set()
genre_diversity = []
novelty = []

# Item popularity from the training dataset (no pandas dependency)
pop_rows = df_train.group_by("item_id").agg(pl.len().alias("n")).iter_rows()
pop_counts = {iid: n for iid, n in pop_rows}
total_events = sum(pop_counts.values())
pop_by_item = {iid: n / total_events for iid, n in pop_counts.items()}

for uid in test_by_user:
    recs = model.predict(train_hist_by_user.get(uid, {}), feats_by_user.get(uid, {}), top_k=TOP_K)
    guids = [g for g, _ in recs]
    recommended_items.update(guids)
    genre_diversity.append(len({item_genre[g] for g in guids}))
    novelty.append(np.mean([-math.log2(max(pop_by_item.get(g, 1e-9), 1e-9)) for g in guids]))

print(f"Coverage@{TOP_K}          = {len(recommended_items)/model.num_items:.3f}  ({len(recommended_items)}/{model.num_items})")
print(f"Intra-list diversity  = {np.mean(genre_diversity):.2f} distinct genres/list  (max={len(GENRES)})")
print(f"Novelty (bits)        = {np.mean(novelty):.2f}")

## 8. Lambda sweep

Lambda (L2 regularization) is usually the single most impactful hyperparameter. We sweep across a few orders of magnitude and plot NDCG@10.

Expect a U-shaped curve: too little λ overfits, too much underfits.

In [ ]:
sweep_lambdas = [1.0, 5.0, 25.0, 100.0, 400.0, 1600.0]
sweep_results = []

for lam in sweep_lambdas:
    m = fease.build_and_train(
        interactions_path=str(i_train_path),
        user_features_path=str(u_path),
        item_features_path=str(t_path),
        alpha=1.0, beta=1.0, lambda_=lam, meta_weight=0.0,
    )
    df = evaluate(m, list(test_by_user.keys()))
    sweep_results.append({"lambda": lam, "ndcg@10": df["ndcg@10"].mean(), "recall@10": df["recall@10"].mean()})
    print(f"lambda={lam:>7.1f}  ndcg@10={sweep_results[-1]['ndcg@10']:.4f}  recall@10={sweep_results[-1]['recall@10']:.4f}")

sweep_df = pl.DataFrame(sweep_results)

In [ ]:
# Plot the sweep (matplotlib is optional — fallback to an ASCII print)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(sweep_df["lambda"].to_list(), sweep_df["ndcg@10"].to_list(), "o-", label="NDCG@10")
    ax.plot(sweep_df["lambda"].to_list(), sweep_df["recall@10"].to_list(), "s--", label="Recall@10")
    ax.set_xscale("log")
    ax.set_xlabel("lambda (L2)")
    ax.set_ylabel("metric")
    ax.set_title("FEASE hyperparameter sweep")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print(sweep_df)

## Summary

What to tune for a new dataset:

1. **`lambda_`** first — sweep across ~3 orders of magnitude and pick the NDCG maximum. Production values for reasonably-sized catalogs usually land between 50 and 200.
2. **`alpha` / `beta`** next — start with 1.0 each. Raise `beta` if cold-start recall is weak; raise `alpha` if item-similarity lists feel incoherent.
3. **Advanced weighting** (`decay_rate`, `ips_alpha`, `event_weights`) — only after a clean baseline. Add one at a time and re-measure.

Continue to `03_deployment_artifact.ipynb` for packaging and qualitative testing.